# Module 6: Speculative Decoding

In Module 5 you made the target model cheaper to serve by moving from BF16 to FP8. Speculative decoding attacks the next part of decode latency: the target model normally verifies one new token at a time. In this module you add a small draft model, `RedHatAI/Qwen3-0.6B-FP8-dynamic`, alongside the FP8 target. The draft model proposes several future tokens, the 4B target verifies them, and accepted tokens let the server move forward by more than one token per target step.


## Learning objectives
- Explain draft-and-verify speculative decoding in plain language
- Distinguish autoregressive draft models, n-gram/suffix methods, native MTP, Medusa, EAGLE, and diffusion-style drafters
- Compute why accepted tokens per target step controls the speedup
- Edit the shared vLLM manifest to add a 0.6B draft model
- Compare the FP8 baseline against speculative decoding with the same load shape
- Decide when speculative decoding helps, hurts, or does nothing


## Prerequisites
- Finished Module 5 and left the vLLM deployment on `RedHatAI/Qwen3-4B-FP8-dynamic`
- The draft model `RedHatAI/Qwen3-0.6B-FP8-dynamic` is pre-cached in the workshop environment
- Your namespace kubeconfig can apply your own `deployment/vllm`
- About 30 minutes


References: [vLLM speculative decoding](https://docs.vllm.ai/en/latest/features/speculative_decoding/) &middot; [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192) &middot; [EAGLE](https://arxiv.org/abs/2401.15077) &middot; [Medusa](https://arxiv.org/abs/2401.10774) &middot; [DFlash](https://arxiv.org/abs/2602.06036)


## Speculative decoding design basics

Normal autoregressive decode asks the target model for one next token at a time. Speculative decoding adds a proposer:

1. The proposer drafts several candidate tokens.
2. The target model verifies those candidates in parallel.
3. The server accepts the prefix the target agrees with and discards the rest.

The target model still decides the final output. The draft path is valuable only when it is cheap enough and accurate enough that the accepted tokens outweigh the extra work.

In this workshop, the proposer is a smaller autoregressive Qwen model. It drafts in a linear chain: token 1, then token 2 conditioned on token 1, and so on, up to a few tokens ahead. That is the most intuitive version here because you can see the target-and-drafter relationship directly.

![A small draft model proposes several tokens, then the FP8 target model verifies the draft and accepts the matching prefix](images/06_speculative_decoding_architecture.svg)


## 1. Setup

Install the small client dependencies, make the repo's `common/` package importable, and resolve the shared manifest path. The manifest is the same repo-root `manifests/vllm.yaml` you edited in Module 5.


In [ ]:

%pip install -q "openai>=1.40" "requests>=2.31"


In [ ]:

# Imports, settings, and paths used throughout the module.
import os, sys
from pathlib import Path

if Path("../manifests/vllm.yaml").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")

sys.path.insert(0, str(REPO_ROOT.resolve()))

from common.config import print_settings
from common import loadtest

settings = print_settings()
MANIFEST = REPO_ROOT / "manifests" / "vllm.yaml"
TARGET_MODEL = "RedHatAI/Qwen3-4B-FP8-dynamic"
DRAFT_MODEL = "RedHatAI/Qwen3-0.6B-FP8-dynamic"
print("manifest:", MANIFEST)
print("target  :", TARGET_MODEL)
print("draft   :", DRAFT_MODEL)


**What you should see:** the FP8 endpoint settings, the shared manifest path, and the target/draft model ids. If `MODEL_NAME` still prints the old BF16 value, that environment variable is stale; the live check in the next section reads the server directly.


## 2. The method map

Speculative decoding is a family of methods. The verifier idea is shared, but the proposer changes.

| Method | Proposer | Best fit | Tradeoff |
|---|---|---|---|
| Draft model | A smaller autoregressive model predicts a linear chain of future tokens | Easy mental model; this workshop path | Extra model memory and acceptance depends on draft-target alignment |
| N-gram lookup | Reuses repeated token sequences from the prompt/history | Repetitive prompts, code, templates | Modest gain; no extra model |
| Suffix decoding | Uses suffix/prefix matching trees over prior text | Agent loops, code, repeated tool traces | More sophisticated pattern lookup; still model-free |
| Native MTP | Extra model heads/checkpoints predict token `t+2`, `t+3`, and beyond | Models trained with multi-token prediction support | Requires model support |
| Medusa | Adds multiple decoding heads to the target model | When Medusa heads are available for the target | Needs trained heads and tree verification |
| EAGLE | Predicts future hidden features, then verifies with the target | Strong general-purpose speculative method | Needs compatible EAGLE weights/integration |
| Diffusion-style, such as DFlash | A lightweight block diffusion drafter proposes a block in parallel | Emerging path for parallel drafting | Newer stack, hardware, and model support |

The vLLM docs expose these through `--speculative-config`. Current vLLM documentation lists methods such as `draft_model`, `ngram`, `suffix`, `mtp`, `eagle3`, and `dflash`. The live exercise uses `draft_model` because the workshop already pre-caches the 0.6B Qwen drafter.


## 3. Acceptance rate, by hand

The key metric is accepted tokens per target step. If the draft proposes four tokens but the target accepts only one, the draft model mostly added overhead. If the target accepts three or four, the target model moves forward several tokens for one expensive verification pass.


In [ ]:

# A tiny acceptance-rate thought experiment. This is not a server benchmark.
target_step_ms = 20
verify_step_ms = 24

for proposed in [2, 4, 6, 8]:
    for acceptance in [0.25, 0.50, 0.75]:
        accepted = proposed * acceptance
        normal_ms_per_token = target_step_ms
        speculative_ms_per_token = verify_step_ms / max(accepted, 1e-9)
        speedup = normal_ms_per_token / speculative_ms_per_token
        print(f"proposed={proposed} acceptance={acceptance:.0%} speedup~{speedup:.2f}x")
    print()


**What you should see:** low acceptance erases the gain, while high acceptance with several proposed tokens can beat normal decode. The exact numbers are illustrative; the shape is the lesson.


## 4. Confirm the FP8 target baseline

Before adding the draft model, confirm the server is still serving the FP8 target from Module 5. Then run a baseline sweep with no speculative decoding.


In [ ]:

# Requires a live vLLM endpoint.
import requests

root = settings.vllm_host.rstrip("/").removesuffix("/v1")

def served_models():
    data = requests.get(
        f"{root}/v1/models",
        headers={"Authorization": f"Bearer {settings.api_key}"},
        timeout=20,
    ).json()
    return [item["id"] for item in data.get("data", [])]

models = served_models()
print("served models:", models)
print("target ok    :", TARGET_MODEL in models)
assert TARGET_MODEL in models, f"Expected {TARGET_MODEL}; got {models}"


**What you should see:** `RedHatAI/Qwen3-4B-FP8-dynamic` in the served-model list. If not, finish Module 5 before continuing.


In [ ]:

# Requires a live vLLM endpoint. Baseline before speculative decoding.
levels = [1, 4, 8, 16]
baseline = loadtest.sweep(levels, input_tokens=256, output_tokens=128, model=TARGET_MODEL)
baseline


**What you should see:** the FP8 target's current tokens per second and TTFT. Speculative decoding is usually most visible at low to medium concurrency, before the GPU is fully saturated, so this sweep starts lower than Module 7's saturation walk.


## 5. Add the draft model

Open `manifests/vllm.yaml` and add one vLLM argument under the existing `args:` list. Put it after the parser flags so it is easy to find later.

```yaml
- '--speculative-config={"method":"draft_model","model":"RedHatAI/Qwen3-0.6B-FP8-dynamic","num_speculative_tokens":4,"draft_tensor_parallel_size":1}'
```

This tells vLLM to run the 0.6B FP8 model as the drafter and let it propose up to four speculative tokens per step. Four is a good workshop starting point: high enough to show the mechanism, low enough to avoid overreaching.


In [ ]:

# Requires a live cluster. Preview the manifest change before applying it.
ns = settings.namespace
!kubectl diff -n {ns} -f {MANIFEST}


In [ ]:

# Requires a live cluster. Apply the edited manifest, wait for the replacement pod,
# and confirm the live Deployment really contains speculative decoding.
import json, subprocess

ns = settings.namespace
!kubectl apply -n {ns} -f {MANIFEST}
!kubectl rollout status -n {ns} deploy/vllm --timeout=10m

raw_args = subprocess.check_output([
    "kubectl", "get", "deployment", "vllm",
    "-o", "jsonpath={.spec.template.spec.containers[0].args}",
], text=True)
live_args = json.loads(raw_args)
spec_args = [arg for arg in live_args if arg.startswith("--speculative-config")]
print("speculative args:", spec_args)
assert spec_args, "Expected --speculative-config in deployment args. Did you edit manifests/vllm.yaml?"
assert DRAFT_MODEL in spec_args[0], f"Expected draft model {DRAFT_MODEL}; got {spec_args[0]}"


**What you should see:** `kubectl diff` should show the added `--speculative-config` argument. `rollout status` should finish successfully, and the live-args check should print the speculative config. If the pod fails to start, check `kubectl logs -n $NAMESPACE deploy/vllm`; common issues are a missing draft model cache, invalid JSON in the argument, or not enough GPU memory for both target and drafter.


## 6. Measure speculative decoding

Run the exact same sweep. The target model, prompt shape, output length, and concurrency levels stay fixed. The only intended change is the draft path.


In [ ]:

# Requires the speculative vLLM pod to be Ready.
speculative = loadtest.sweep(levels, input_tokens=256, output_tokens=128, model=TARGET_MODEL)
speculative


In [ ]:

# Compare baseline and speculative decoding side by side.
print("concurrency | base tok/s | spec tok/s | base p95 TTFT | spec p95 TTFT")
for before, after in zip(baseline, speculative):
    print(f"{before['concurrency']:>11} | {before['throughput_tok_s']:>10} | {after['throughput_tok_s']:>10} | "
          f"{before['ttft_p95_ms']:>13} | {after['ttft_p95_ms']:>13}")

best_base = max(row["throughput_tok_s"] for row in baseline)
best_spec = max(row["throughput_tok_s"] for row in speculative)
print(f"peak throughput: {best_base:.1f} -> {best_spec:.1f} tok/s ({best_spec / best_base:.2f}x)")


**What you should see:** a before/after table. The gain may be strongest at low to medium concurrency. At high concurrency, the extra draft work can compete with normal batching, so speculative decoding is not automatically better for every load level.


## 7. Tune the speculation depth

`num_speculative_tokens` is the first knob. If it is too low, you leave possible speedup on the table. If it is too high, rejected draft tokens waste work. Try `2`, `4`, and `6` only if there is time. Keep the same sweep each time.


In [ ]:

# Optional: after editing num_speculative_tokens in the manifest, re-apply and rerun.
# Good values to try here: 2, 4, 6.
# !kubectl diff -n {settings.namespace} -f {MANIFEST}
# !kubectl apply -n {settings.namespace} -f {MANIFEST}
# !kubectl rollout status -n {settings.namespace} deploy/vllm --timeout=10m
# tuned = loadtest.sweep(levels, input_tokens=256, output_tokens=128, model=TARGET_MODEL)


**What you should see:** the best speculation depth is workload-dependent. Predictable outputs can accept longer drafts. Unpredictable outputs usually need shorter drafts.


## Things to know

- **The target still verifies.** Proper speculative decoding preserves the target model's output distribution; the drafter does not get to unilaterally answer.
- **Draft-target alignment matters.** A small model from the same family and tokenizer is a better drafter than a random small model.
- **Memory can become the blocker.** The draft model consumes GPU memory that could otherwise be KV cache.
- **Low to medium QPS is the sweet spot.** When the GPU is already saturated, extra drafting can reduce or erase the gain.
- **Other methods exist for good reasons.** N-gram and suffix avoid a second model; MTP, Medusa, EAGLE, and DFlash need trained auxiliary heads or models but can draft more effectively.


## Try it yourself

**Change speculation depth.** Try `num_speculative_tokens` of `2`, `4`, and `6`. Which one gives the best throughput without hurting TTFT?

**Predict by prompt family.** For summarization, code completion, strict JSON, tool choice, and creative writing, predict whether draft acceptance should be high or low. Then explain which speculative method you would try first.


## Summary

- Speculative decoding tries to move the target model forward by more than one token per verification step.
- The workshop path uses a 0.6B FP8 autoregressive draft model alongside the 4B FP8 target.
- Acceptance rate and draft overhead decide whether the speedup appears.
- vLLM exposes speculative decoding through `--speculative-config`; this module adds that config to the shared manifest.
- Leave the speculative deployment running. Module 7 drives load against the optimized path and finds the next bottleneck.


## Next

**Module 7: Engine Mechanics and Saturation.** Quantization and speculative decoding lowered the per-token cost. Next you push the serving engine until batching, queueing, and saturation show up in the metrics.
